# CalCOFI DIC 回帰のハイパーパラメータ探索

各設定を3つの初期化シードで学習し、最終エポックの検証MSEの平均を Optuna の TPE で最小化します。試行結果、上位設定、グラフは `output/optuna/` に保存します。

In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
from joblib import Parallel, delayed, parallel_config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA_DIR = Path("dataset")
OUTPUT_DIR = Path("output/optuna")

# データ分割は固定し、各設定を同じ3つのモデルシードで比較する。
SPLIT_SEED = 42
SEARCH_SEED = 42
MODEL_SEEDS = (42, 123, 2026)

MAX_EPOCHS = 5000
MIN_LR = 1e-6
N_TRIALS = 32
N_STARTUP_TRIALS = 8  # 基準設定を含む、TPE が探索履歴を使い始めるまでの試行数
N_WORKERS = 4
N_FINALISTS = 10  # 基準設定に加えて比較表に載せる件数

# Huber の閾値は標準化前の DIC の尺度で指定する。
CATEGORICAL_CHOICES = {
    "hidden1": (128, 256),
    "hidden2": (32, 64, 128),
    "celu_alpha": (2.5, 3, 3.5, 4),
    "huber_delta": (5, 10),
    "batch_size": (32, 64),
}
NUMERIC_RANGES = {
    "learning_rate": (5e-4, 1.5e-3),
    "weight_decay": (0.01, 0.04),
}

# 現行の提出用ノートブックの設定を、比較の基準として試行0で評価する。
BASELINE = {
    "hidden1": 256,
    "hidden2": 64,
    "celu_alpha": 2.5,
    "learning_rate": 0.001,
    "weight_decay": 0.025,
    "huber_delta": 10,
    "batch_size": 64,
}

### データ分割と前処理

提出用ノートブックと同じ15列を使います。入力と DIC の標準化には訓練分割の統計量を使い、その値で検証分割も変換します。

In [ ]:
train_csv = DATA_DIR / "train.csv"
train = pd.read_csv(train_csv)
test_columns = pd.read_csv(DATA_DIR / "test.csv", nrows=0).columns
feature_columns = [
    column for column in train.columns
    if column in test_columns and column not in {"id", "DIC", "Unnamed: 12"}
]
X_train, X_val, y_train, y_val = train_test_split(
    train[feature_columns],
    train["DIC"],
    test_size=0.2,
    random_state=SPLIT_SEED,
)

# 入力は列ごとに、DIC は目的変数として標準化する。
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

target_mean = y_train.mean()
target_std = y_train.std(ddof=0)
y_train_scaled = ((y_train - target_mean) / target_std).to_numpy()
y_val_scaled = ((y_val - target_mean) / target_std).to_numpy()

print(f"訓練 {len(y_train)} 行、検証 {len(y_val)} 行、入力 {len(feature_columns)} 列")

In [ ]:
def fit_and_score(params, seed, X_train, X_val, y_train_scaled, y_val_scaled, target_std):
    """1つの設定・シードで学習し、元の DIC 尺度で最終検証MSEを返す。"""
    torch.set_num_threads(1)  # 並列ワーカーごとにCPUを1スレッド使う。
    torch.use_deterministic_algorithms(True)
    torch.manual_seed(seed)

    train_data = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train_scaled, dtype=torch.float32),
    )
    train_loader = DataLoader(
        train_data,
        batch_size=params["batch_size"],
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    val_features = torch.tensor(X_val, dtype=torch.float32)
    val_targets = torch.tensor(y_val_scaled, dtype=torch.float32)

    model = nn.Sequential(
        nn.Linear(X_train.shape[1], params["hidden1"]),
        nn.CELU(alpha=params["celu_alpha"]),
        nn.Linear(params["hidden1"], params["hidden2"]),
        nn.CELU(alpha=params["celu_alpha"]),
        nn.Linear(params["hidden2"], 1),
    )
    # 誤差を標準化しても、Huber の閾値が元の DIC 尺度で同じになるようにする。
    criterion = nn.HuberLoss(delta=params["huber_delta"] / target_std)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params["learning_rate"], weight_decay=params["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LR,
    )

    model.train()
    for _ in range(MAX_EPOCHS):
        for features, targets in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(features).squeeze(-1), targets)
            loss.backward()
            optimizer.step()
        scheduler.step()

    model.eval()
    with torch.inference_mode():
        predictions = model(val_features).squeeze(-1)
        scaled_mse = nn.functional.mse_loss(predictions, val_targets).item()
    return scaled_mse * target_std**2


def suggest_params(trial):
    """離散値と対数スケールの範囲から候補を提案する。"""
    params = {
        name: trial.suggest_categorical(name, choices)
        for name, choices in CATEGORICAL_CHOICES.items()
    }
    params.update({
        name: trial.suggest_float(name, lower, upper, log=True)
        for name, (lower, upper) in NUMERIC_RANGES.items()
    })
    return params

### 候補を探索する

試行0に基準設定を入れます。候補をまとめて提案し、各候補を3シードで並列学習します。各シードの検証MSEを保存し、その平均を試行番号順に Optuna に返します。

In [ ]:
sampler = optuna.samplers.TPESampler(
    seed=SEARCH_SEED,
    n_startup_trials=N_STARTUP_TRIALS,
    multivariate=True,
)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.enqueue_trial(BASELINE)

model_data = (X_train, X_val, y_train_scaled, y_val_scaled, target_std)
seed_columns = [f"val_mse_seed_{seed}" for seed in MODEL_SEEDS]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with (
    parallel_config(backend="loky", n_jobs=N_WORKERS, inner_max_num_threads=1),
    Parallel() as run_models,
):
    for start in range(0, N_TRIALS, N_WORKERS):
        n_candidates = min(N_WORKERS, N_TRIALS - start)
        trials = [study.ask() for _ in range(n_candidates)]
        candidates = [suggest_params(trial) for trial in trials]
        scores = run_models(
            delayed(fit_and_score)(params, seed, *model_data)
            for params in candidates
            for seed in MODEL_SEEDS
        )

        # 並列学習の結果を、1行が1試行、1列が1シードになるように並べる。
        scores_by_trial = np.asarray(scores).reshape(n_candidates, len(MODEL_SEEDS))
        for trial, seed_scores in zip(trials, scores_by_trial, strict=True):
            for column, score in zip(seed_columns, seed_scores, strict=True):
                trial.set_user_attr(column, float(score))
            mean_mse = float(seed_scores.mean())
            study.tell(trial, mean_mse)
            print(f"試行 {trial.number:02d} | 平均検証MSE {mean_mse:.6f}", flush=True)

        # 学習の長い探索でも、完了した試行はバッチごとに保存する。
        study.trials_dataframe(
            attrs=("number", "value", "params", "state", "user_attrs"),
        ).to_csv(OUTPUT_DIR / "trials.csv", index=False)

### 上位設定を比較する

3シードの平均MSEで順位を付け、上位設定と基準設定のシード別MSEおよびばらつきを表にします。

In [ ]:
completed_trials = study.get_trials(
    deepcopy=False, states=(optuna.trial.TrialState.COMPLETE,),
)
ranked_trials = sorted(completed_trials, key=lambda trial: trial.value)
top_trials = [trial for trial in ranked_trials if trial.number != 0][:N_FINALISTS]
finalists = [trial for trial in ranked_trials if trial.number == 0 or trial in top_trials]

# 探索時に保存したシード別MSEを使って、比較表を作る。
results = []
for trial in finalists:
    seed_scores = {column: trial.user_attrs[column] for column in seed_columns}
    results.append({
        "trial": trial.number,
        **trial.params,
        **seed_scores,
        "mean_final_val_mse": trial.value,
        "std_final_val_mse": float(np.std(list(seed_scores.values()), ddof=1)),
    })
finalist_results = pd.DataFrame(results)
finalist_results.to_csv(OUTPUT_DIR / "finalists.csv", index=False)

adopted_trial = study.best_trial
adopted = results[0]
summary = {
    "optuna_version": optuna.__version__,
    "torch_version": torch.__version__,
    "sampler": "multivariate TPE",
    "trials": N_TRIALS,
    "startup_trials": N_STARTUP_TRIALS,
    "workers": N_WORKERS,
    "search_seed": SEARCH_SEED,
    "model_seeds": MODEL_SEEDS,
    "split_seed": SPLIT_SEED,
    "epochs": MAX_EPOCHS,
    "cosine_eta_min": MIN_LR,
    "scaler_fit_rows": int(scaler.n_samples_seen_),
    "target_mean": float(target_mean),
    "target_std": float(target_std),
    "data_sha256": hashlib.sha256(train_csv.read_bytes()).hexdigest(),
    "adopted_trial": adopted_trial.number,
    "adopted_params": adopted_trial.params,
    "adopted_mean_final_val_mse": adopted["mean_final_val_mse"],
    "adopted_std_final_val_mse": adopted["std_final_val_mse"],
    **{
        f"adopted_seed_{seed}_final_val_mse": adopted_trial.user_attrs[f"val_mse_seed_{seed}"]
        for seed in MODEL_SEEDS
    },
}
(OUTPUT_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8",
)
finalist_results

### パラメータ別の検証MSEと重要度

各点は1設定の3シード平均MSEで、赤いひし形は平均MSEが最低だった設定です。パラメータの重要度は同じ試行結果から fANOVA で推定します。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
from optuna.importance import FanovaImportanceEvaluator, get_param_importances

trial_mses = [trial.value for trial in completed_trials]
parameter_labels = {
    "hidden1": "Hidden layer 1",
    "hidden2": "Hidden layer 2",
    "celu_alpha": "CELU alpha",
    "learning_rate": "Initial learning rate",
    "weight_decay": "Weight decay",
    "huber_delta": "Huber delta",
    "batch_size": "Training batch size",
}

fig, axes = plt.subplots(4, 2, figsize=(11, 11), sharey=True, layout="constrained")
for ax, (name, label) in zip(axes.flat, parameter_labels.items()):
    parameter_values = [trial.params[name] for trial in completed_trials]
    adopted_value = adopted_trial.params[name]
    if name in CATEGORICAL_CHOICES:
        choices = list(CATEGORICAL_CHOICES[name])
        positions = {choice: index for index, choice in enumerate(choices)}
        parameter_values = [positions[value] for value in parameter_values]
        adopted_value = positions[adopted_value]
        ax.set_xticks(range(len(choices)), labels=[str(choice) for choice in choices])
        ax.set_xlim(-0.35, len(choices) - 0.65)
    else:
        ax.set_xscale("log")
        lower, upper = NUMERIC_RANGES[name]
        ax.set_xlim(lower, upper)
        ax.set_xticks([lower, upper], labels=[f"{lower:g}", f"{upper:g}"])
        ax.xaxis.set_minor_formatter(NullFormatter())

    trials_artist = ax.scatter(
        parameter_values, trial_mses,
        color="#2563a0", alpha=0.52, s=24, label="Completed study trial",
    )
    adopted_artist = ax.scatter(
        adopted_value, adopted_trial.value, color="#bd4a26", edgecolors="white",
        linewidths=0.7, marker="D", s=66, zorder=3, label="Adopted setting in study",
    )
    ax.set_title(label, loc="left")
    ax.set_yscale("log")
    ax.grid(axis="y", alpha=0.25)

axes[-1, -1].axis("off")
axes[-1, -1].legend(handles=[trials_artist, adopted_artist], loc="center", frameon=False)
fig.suptitle("Optuna trial results by hyperparameter", fontsize=16)
fig.supylabel("Mean validation MSE across model seeds (log scale)")
fig.savefig(OUTPUT_DIR / "parameter_slices.png", dpi=160)
plt.show()
plt.close(fig)

importances = get_param_importances(study, evaluator=FanovaImportanceEvaluator(seed=SEARCH_SEED))
names = list(importances)
scores = [importances[name] for name in names]

fig, ax = plt.subplots(figsize=(8, 4.6), layout="constrained")
ax.barh([parameter_labels[name] for name in names], scores, color="#2563a0")
ax.invert_yaxis()
ax.set_xlim(0, min(1.1, 1.18 * max(scores)))
ax.bar_label(ax.containers[0], fmt="%.3f", padding=4)
ax.set_title("Optuna hyperparameter importance (fANOVA)", loc="left")
ax.set_xlabel("Normalized importance")
ax.set_axisbelow(True)
ax.grid(axis="x", alpha=0.25)
fig.savefig(OUTPUT_DIR / "parameter_importance.png", dpi=160)
plt.show()
plt.close(fig)